# SigLIP-2 Fake News Detection — Fakeddit Dataset
**Vision-Language Model with Sigmoid Loss for Binary Classification**

## Model: SigLIP-2 (google/siglip-base-patch16-224)
- **Architecture**: Sigmoid Loss for Language-Image Pre-training
- **Dataset**: Fakeddit (5000 samples)
- **Task**: 2-way binary classification (Real vs Fake)
- **Note**: 6-way and 3-way classifications are **disabled** (binary only)

## Configuration:
- **5-Fold Cross-Validation** (equal splits)
- **80% train / 10% val / 10% test** per fold
- **10 epochs** with early stopping
- **Anti-overfitting**: dropout, weight decay, label smoothing, early stopping

## Output (per fold + mean):
✅ Accuracy, F1-Score, Precision, Recall

✅ **Overfitting Analysis**: Train-Val gap, Train-Test gap, Verdict

In [ ]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
import sys, subprocess

print("Installing SigLIP and dependencies...\n")
packages = ["transformers>=4.35.0", "accelerate", "pillow", "tabulate", "scikit-learn"]

for i, pkg in enumerate(packages, 1):
    print(f"[{i}/{len(packages)}] {pkg}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\n✓ Installation complete!\n")

# Verify
from transformers import AutoProcessor, AutoModel
import torch
print(f"✓ Transformers {__import__('transformers').__version__}")
print(f"✓ PyTorch {torch.__version__}")

In [ ]:
# ── Cell 2: Imports & Config ──────────────────────────────────────────────────
import os, gc, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoProcessor, AutoModel
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tabulate import tabulate
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

# ════════════════════════ CONFIGURATION ══════════════════════════════════════
SEED=42; NUM_SAMPLES=5000; NUM_FOLDS=5; EPOCHS=10; BATCH_SIZE=24
LEARNING_RATE=2e-5; WEIGHT_DECAY=1e-2; DROPOUT_RATE=0.3; PATIENCE=3
NUM_CLASSES=2  # 2-way ONLY (6-way and 3-way DISABLED)
MAX_TEXT_LEN=64; DEVICE="cuda" if torch.cuda.is_available() else "cpu"
SIGLIP_MODEL="google/siglip-base-patch16-224"; OUTPUT_DIR="/kaggle/working"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"SigLIP-2 | {EPOCHS} epochs | {NUM_FOLDS} folds | Batch {BATCH_SIZE}")
print(f"Task: 2-way binary (6-way & 3-way DISABLED)")
print("="*70)

In [ ]:
# ── Cell 3: Fakeddit Loader (2-way ONLY) ─────────────────────────────────────

def load_fakeddit(n_samples=NUM_SAMPLES):
    """
    Load Fakeddit dataset with 2-way labels ONLY.
    6-way and 3-way classifications are SKIPPED.
    """
    print("🔍 Scanning for Fakeddit dataset...\n")
    
    # Find TSV files
    for tsv in Path("/kaggle/input").rglob("*.tsv"):
        if "sample" in tsv.name.lower() or "train" in tsv.name.lower():
            print(f"✓ Found: {tsv.name}")
            return load_from_tsv(tsv, n_samples)
    
    # Fallback to CSV
    for csv in Path("/kaggle/input").rglob("*.csv"):
        print(f"✓ Found: {csv.name}")
        return load_from_tsv(csv, n_samples)
    
    raise FileNotFoundError("❌ No Fakeddit TSV/CSV found!")


def load_from_tsv(tsv_path, n_samples):
    """Load Fakeddit TSV with 2-way labels only."""
    sep = "\t" if tsv_path.suffix == ".tsv" else ","
    df = pd.read_csv(tsv_path, sep=sep, low_memory=False)
    df.columns = df.columns.str.strip()
    
    print(f"Raw rows: {len(df)}")
    print(f"Columns: {list(df.columns[:10])}...")
    
    # Find columns
    text_col = next((c for c in ["clean_title", "title", "text"] if c in df.columns), None)
    label_col = next((c for c in ["2_way_label", "label"] if c in df.columns), None)
    id_col = "id" if "id" in df.columns else df.columns[0]
    
    if not text_col:
        raise ValueError(f"No text column found. Available: {list(df.columns[:15])}")
    if not label_col:
        raise ValueError(f"No 2_way_label column found. Available: {list(df.columns[:15])}")
    
    # Extract relevant columns
    df = df[[id_col, text_col, label_col]].dropna()
    df.columns = ["id", "text", "label"]
    df["label"] = df["label"].astype(int)
    
    # Filter: 2-way only (0=real, 1=fake)
    df = df[df["label"].isin([0, 1])].reset_index(drop=True)
    print(f"\nAfter filtering to 2-way: {len(df)} rows")
    print(f"  Real (0): {(df['label']==0).sum()} | Fake (1): {(df['label']==1).sum()}")
    
    # Find images
    img_dir = tsv_path.parent / "public_image_set"
    if img_dir.exists():
        print(f"  Image dir: {img_dir}")
        df["image_path"] = df["id"].apply(lambda x: find_image(x, img_dir))
        df = df[df["image_path"].notna()]
        print(f"  With images: {len(df)}")
    else:
        print(f"  ⚠ No image directory found, using text-only placeholders")
        df["image_path"] = None
    
    # Balance
    half = n_samples // 2
    fake = df[df["label"]==1].sample(min(half, (df["label"]==1).sum()), random_state=SEED)
    real = df[df["label"]==0].sample(min(half, (df["label"]==0).sum()), random_state=SEED)
    actual = min(len(fake), len(real))
    balanced = pd.concat([fake.iloc[:actual], real.iloc[:actual]])
    balanced = balanced.sample(frac=1, random_state=SEED).reset_index(drop=True)
    
    print(f"\n✓ Final: {len(balanced)} balanced samples")
    print(f"  Real: {(balanced['label']==0).sum()} | Fake: {(balanced['label']==1).sum()}")
    
    return balanced


def find_image(img_id, img_dir):
    for ext in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
        path = img_dir / f"{img_id}{ext}"
        if path.exists(): return str(path)
    return None


print("Fakeddit loader ready ✓")

In [ ]:
# ── Cell 4: SigLIP Dataset ────────────────────────────────────────────────────

class SigLIPDataset(Dataset):
    def __init__(self, df, processor, max_length=MAX_TEXT_LEN):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length
        self.placeholder = Image.new("RGB", (224, 224), (127, 127, 127))
    
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text"])
        label = int(row["label"])
        
        # Load image
        img_path = row.get("image_path")
        try:
            img = Image.open(img_path).convert("RGB") if img_path else self.placeholder
        except:
            img = self.placeholder
        
        # Process
        enc = self.processor(images=img, text=text, padding="max_length",
                            truncation=True, max_length=self.max_length, return_tensors="pt")
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        enc["labels"] = torch.tensor(label, dtype=torch.long)
        return enc


def collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch])
    }


print("Dataset class ready ✓")

In [ ]:
# ── Cell 5: SigLIP Classifier ─────────────────────────────────────────────────

class SigLIPClassifier(nn.Module):
    def __init__(self, siglip_model, num_classes=2, dropout=DROPOUT_RATE):
        super().__init__()
        self.siglip = siglip_model
        self.vision_dim = siglip_model.config.vision_config.hidden_size
        self.text_dim = siglip_model.config.text_config.hidden_size
        
        # Freeze all
        for p in self.siglip.parameters(): p.requires_grad = False
        
        # Unfreeze last 2 vision layers
        if hasattr(self.siglip.vision_model, 'encoder'):
            for layer in self.siglip.vision_model.encoder.layers[-2:]:
                for p in layer.parameters(): p.requires_grad = True
        
        # Head
        fused_dim = self.vision_dim + self.text_dim
        hidden = fused_dim * 2
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim), nn.Linear(fused_dim, hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Dropout(dropout/2), nn.Linear(hidden//2, num_classes)
        )
    
    def forward(self, pixel_values, input_ids, attention_mask):
        out = self.siglip(pixel_values=pixel_values, input_ids=input_ids,
                         attention_mask=attention_mask, return_dict=True)
        vision = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
        text = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
        fused = torch.cat([vision, text], dim=1)
        return self.classifier(fused)


print("Model defined ✓")

In [ ]:
# ── Cell 6: Training Helpers ──────────────────────────────────────────────────

class EarlyStopping:
    def __init__(self, patience=PATIENCE, delta=1e-4):
        self.patience, self.delta = patience, delta
        self.best_score, self.counter, self.best_state, self.stop = None, 0, None, False
    def step(self, val_loss, model):
        score = -val_loss
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score, self.counter = score, 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience: self.stop = True
    def restore_best(self, model):
        if self.best_state: model.load_state_dict({k: v.to(DEVICE) for k, v in self.best_state.items()})


def train_epoch(model, loader, optimizer, criterion, scheduler):
    model.train(); total_loss = correct = n = 0
    for batch in loader:
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        logits = model(**batch); loss = criterion(logits, labels)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item(); n += labels.size(0)
    scheduler.step()
    return total_loss/n, correct/n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); total_loss = 0.0; all_preds, all_labels = [], []
    for batch in loader:
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch); loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    n = len(all_labels)
    return dict(loss=total_loss/n, accuracy=accuracy_score(all_labels, all_preds),
                f1=f1_score(all_labels, all_preds, average="weighted", zero_division=0),
                precision=precision_score(all_labels, all_preds, average="weighted", zero_division=0),
                recall=recall_score(all_labels, all_preds, average="weighted", zero_division=0)), \
           np.array(all_preds), np.array(all_labels)


print("Training helpers ready ✓")

In [ ]:
# ── Cell 7: K-Fold Loop ───────────────────────────────────────────────────────

def run_kfold(df):
    skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    labels, indices = df["label"].values, np.arange(len(df))
    fold_results, all_history, all_cms, overfitting_stats = [], [], [], []
    
    processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
    print("✓ SigLIP processor loaded\n")
    
    for fold, (train_val_idx, test_idx) in enumerate(skf.split(indices, labels), 1):
        print(f"{'='*70}\n  FOLD {fold}/{NUM_FOLDS}\n{'='*70}")
        
        # 80/10/10 split
        np.random.shuffle(train_val_idx); mid = len(test_idx)//2
        val_idx, test_idx_f, train_idx = test_idx[:mid], test_idx[mid:], train_val_idx
        print(f"  Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx_f)}")
        
        # DataLoaders
        nw = 2 if DEVICE=="cuda" else 0
        train_l = DataLoader(SigLIPDataset(df.iloc[train_idx], processor), BATCH_SIZE, True,
                            collate_fn=collate_fn, num_workers=nw, pin_memory=(DEVICE=="cuda"))
        val_l = DataLoader(SigLIPDataset(df.iloc[val_idx], processor), BATCH_SIZE, False,
                          collate_fn=collate_fn, num_workers=nw, pin_memory=(DEVICE=="cuda"))
        test_l = DataLoader(SigLIPDataset(df.iloc[test_idx_f], processor), BATCH_SIZE, False,
                           collate_fn=collate_fn, num_workers=nw, pin_memory=(DEVICE=="cuda"))
        
        # Model
        print("  Loading SigLIP...")
        siglip_base = AutoModel.from_pretrained(SIGLIP_MODEL)
        model = SigLIPClassifier(siglip_base, NUM_CLASSES, DROPOUT_RATE).to(DEVICE)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        optimizer = AdamW(model.parameters(), LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, EPOCHS, 1e-6)
        early_stop = EarlyStopping(PATIENCE)
        history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}
        
        # Training
        for epoch in range(1, EPOCHS+1):
            tr_loss, tr_acc = train_epoch(model, train_l, optimizer, criterion, scheduler)
            val_m, _, _ = evaluate(model, val_l, criterion)
            history["train_loss"].append(tr_loss); history["val_loss"].append(val_m["loss"])
            history["train_acc"].append(tr_acc); history["val_acc"].append(val_m["accuracy"])
            early_stop.step(val_m["loss"], model)
            print(f"  E{epoch:2d}/{EPOCHS} | TrL={tr_loss:.4f} TrA={tr_acc:.4f} | VaL={val_m['loss']:.4f} VaA={val_m['accuracy']:.4f}" + ("  ← best" if early_stop.counter==0 else ""))
            if early_stop.stop: print(f"  Early stop at E{epoch}"); break
        
        early_stop.restore_best(model)
        
        # Evaluate
        train_final, _, _ = evaluate(model, train_l, criterion)
        val_final, _, _ = evaluate(model, val_l, criterion)
        test_m, preds, true = evaluate(model, test_l, criterion)
        cm = confusion_matrix(true, preds); all_cms.append(cm)
        
        # Overfitting
        tv_gap = train_final["accuracy"] - val_final["accuracy"]
        tt_gap = train_final["accuracy"] - test_m["accuracy"]
        gen = (val_final["accuracy"] + test_m["accuracy"]) / 2
        overfitting_stats.append({"Fold":fold, "Train_Acc":round(train_final["accuracy"],4),
                                  "Val_Acc":round(val_final["accuracy"],4), "Test_Acc":round(test_m["accuracy"],4),
                                  "Train-Val_Gap":round(tv_gap,4), "Train-Test_Gap":round(tt_gap,4), "Generalization":round(gen,4)})
        
        print(f"\n  ── Fold {fold} Test ──")
        print(f"  Acc={test_m['accuracy']:.4f} F1={test_m['f1']:.4f} Prec={test_m['precision']:.4f} Rec={test_m['recall']:.4f}")
        print(f"  🔍 Overfitting: TV_gap={tv_gap:.4f} {'⚠️' if tv_gap>0.1 else '✓'} | TT_gap={tt_gap:.4f} {'⚠️' if tt_gap>0.1 else '✓'}\n")
        print(classification_report(true, preds, target_names=["Real","Fake"]))
        
        fold_results.append({"Fold":fold, "Accuracy":round(test_m["accuracy"],4), "F1-Score":round(test_m["f1"],4),
                            "Precision":round(test_m["precision"],4), "Recall":round(test_m["recall"],4)})
        all_history.append(history)
        del model, siglip_base; gc.collect()
        if DEVICE=="cuda": torch.cuda.empty_cache()
    
    return fold_results, all_history, all_cms, overfitting_stats


print("K-fold runner ready ✓")

In [ ]:
# ── Cell 8: Results Display ───────────────────────────────────────────────────

def print_results(fold_results, overfitting_stats):
    print("\n"+"="*70+"\n  📊 PERFORMANCE METRICS\n"+"="*70)
    rows = [[r["Fold"],r["Accuracy"],r["F1-Score"],r["Precision"],r["Recall"]] for r in fold_results]
    accs,f1s,precs,recs = [r["Accuracy"] for r in fold_results],[r["F1-Score"] for r in fold_results],[r["Precision"] for r in fold_results],[r["Recall"] for r in fold_results]
    rows.append(["Mean±Std", f"{np.mean(accs):.4f}±{np.std(accs):.4f}", f"{np.mean(f1s):.4f}±{np.std(f1s):.4f}",
                 f"{np.mean(precs):.4f}±{np.std(precs):.4f}", f"{np.mean(recs):.4f}±{np.std(recs):.4f}"])
    print(tabulate(rows, headers=["Fold","Accuracy","F1-Score","Precision","Recall"], tablefmt="fancy_grid"))
    
    print("\n"+"="*70+"\n  🔍 OVERFITTING ANALYSIS\n"+"="*70)
    overfit_rows = [[s["Fold"],s["Train_Acc"],s["Val_Acc"],s["Test_Acc"],s["Train-Val_Gap"],s["Train-Test_Gap"],s["Generalization"]] for s in overfitting_stats]
    tr,va,te,tvg,ttg,gen = [s["Train_Acc"] for s in overfitting_stats],[s["Val_Acc"] for s in overfitting_stats],[s["Test_Acc"] for s in overfitting_stats],[s["Train-Val_Gap"] for s in overfitting_stats],[s["Train-Test_Gap"] for s in overfitting_stats],[s["Generalization"] for s in overfitting_stats]
    overfit_rows.append(["Mean", f"{np.mean(tr):.4f}", f"{np.mean(va):.4f}", f"{np.mean(te):.4f}", f"{np.mean(tvg):.4f}", f"{np.mean(ttg):.4f}", f"{np.mean(gen):.4f}"])
    print(tabulate(overfit_rows, headers=["Fold","Train_Acc","Val_Acc","Test_Acc","Train-Val_Gap","Train-Test_Gap","Generalization"], tablefmt="fancy_grid"))
    
    mean_tv, mean_tt = np.mean(tvg), np.mean(ttg)
    verdict = "✓ EXCELLENT" if mean_tv<0.05 and mean_tt<0.05 else "✓ GOOD" if mean_tv<0.10 and mean_tt<0.10 else "⚠ MODERATE" if mean_tv<0.15 else "⚠ HIGH"
    print(f"\n{'='*70}\n  📈 VERDICT: {verdict}\n  Train-Val Gap: {mean_tv:.4f} | Train-Test Gap: {mean_tt:.4f}\n{'='*70}")
    return {"mean_accuracy":np.mean(accs), "mean_f1":np.mean(f1s), "mean_tv_gap":mean_tv, "verdict":verdict}


def save_results(fold_results, overfitting_stats):
    df_perf = pd.DataFrame(fold_results)
    accs,f1s,precs,recs = df_perf["Accuracy"],df_perf["F1-Score"],df_perf["Precision"],df_perf["Recall"]
    df_perf = pd.concat([df_perf, pd.DataFrame([{"Fold":"Mean±Std", "Accuracy":f"{accs.mean():.4f}±{accs.std():.4f}",
                                                 "F1-Score":f"{f1s.mean():.4f}±{f1s.std():.4f}",
                                                 "Precision":f"{precs.mean():.4f}±{precs.std():.4f}",
                                                 "Recall":f"{recs.mean():.4f}±{recs.std():.4f}"}])])
    df_perf.to_csv(f"{OUTPUT_DIR}/siglip_fakeddit_performance.csv", index=False)
    
    df_over = pd.DataFrame(overfitting_stats)
    df_over = pd.concat([df_over, pd.DataFrame([{"Fold":"Mean", "Train_Acc":f"{df_over['Train_Acc'].mean():.4f}",
                                                 "Val_Acc":f"{df_over['Val_Acc'].mean():.4f}",
                                                 "Test_Acc":f"{df_over['Test_Acc'].mean():.4f}",
                                                 "Train-Val_Gap":f"{df_over['Train-Val_Gap'].mean():.4f}",
                                                 "Train-Test_Gap":f"{df_over['Train-Test_Gap'].mean():.4f}",
                                                 "Generalization":f"{df_over['Generalization'].mean():.4f}"}])])
    df_over.to_csv(f"{OUTPUT_DIR}/siglip_fakeddit_overfitting.csv", index=False)
    
    print(f"\n✓ Saved: siglip_fakeddit_performance.csv")
    print(f"✓ Saved: siglip_fakeddit_overfitting.csv")
    print("\n📊 Performance:"); display(df_perf)
    print("\n🔍 Overfitting:"); display(df_over)


def plot_viz(all_history, all_cms):
    fig,axes=plt.subplots(2,NUM_FOLDS,figsize=(5*NUM_FOLDS,8)); C=["#2196F3","#FF5722"]
    for i,h in enumerate(all_history):
        ep=range(1,len(h["train_loss"])+1)
        axes[0,i].plot(ep,h["train_loss"],color=C[0],lw=2); axes[0,i].plot(ep,h["val_loss"],color=C[1],lw=2,ls="--")
        axes[0,i].set_title(f"F{i+1} Loss",fontweight="bold"); axes[0,i].grid(alpha=0.3)
        axes[1,i].plot(ep,h["train_acc"],color=C[0],lw=2); axes[1,i].plot(ep,h["val_acc"],color=C[1],lw=2,ls="--")
        axes[1,i].set_title(f"F{i+1} Acc",fontweight="bold"); axes[1,i].set_ylim(0,1); axes[1,i].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/training_curves.png",dpi=150); plt.show()
    
    fig,axes=plt.subplots(1,NUM_FOLDS,figsize=(5*NUM_FOLDS,5))
    for i,cm in enumerate(all_cms):
        sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=axes[i],xticklabels=["Real","Fake"],yticklabels=["Real","Fake"])
        axes[i].set_title(f"F{i+1}",fontweight="bold")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/confusion_matrices.png",dpi=150); plt.show()
    print(f"\n✓ Plots saved")


print("Results helpers ready ✓")

In [ ]:
# ── Cell 9: Load Fakeddit ─────────────────────────────────────────────────────
print("Loading Fakeddit (2-way only)...\n")
df_meta = load_fakeddit(NUM_SAMPLES)
print("\nSample rows:")
df_meta.head(3)

In [ ]:
# ── Cell 10: Run Training ─────────────────────────────────────────────────────
# ⚠ Takes ~30-45 min on Kaggle GPU T4
print("\nStarting 5-fold CV...\n")
fold_results, all_history, all_cms, overfitting_stats = run_kfold(df_meta)

In [ ]:
# ── Cell 11: Display & Save Results ───────────────────────────────────────────
summary = print_results(fold_results, overfitting_stats)
save_results(fold_results, overfitting_stats)

In [ ]:
# ── Cell 12: Visualizations ───────────────────────────────────────────────────
plot_viz(all_history, all_cms)

In [ ]:
# ── Cell 13: Final Summary ────────────────────────────────────────────────────
print("\n"+"="*70+"\n  🎯 FINAL SUMMARY\n"+"="*70)
print(f"  Model: SigLIP-2 (google/siglip-base-patch16-224)")
print(f"  Dataset: Fakeddit ({NUM_SAMPLES} samples, 2-way ONLY)")
print(f"  6-way & 3-way: DISABLED ✓")
print(f"\n  📊 Performance:")
print(f"     Mean Accuracy : {summary['mean_accuracy']:.4f}")
print(f"     Mean F1-Score : {summary['mean_f1']:.4f}")
print(f"\n  🔍 Overfitting:")
print(f"     Train-Val Gap: {summary['mean_tv_gap']:.4f}")
print(f"     Verdict      : {summary['verdict']}")
print(f"\n  💾 Files: siglip_fakeddit_performance.csv, siglip_fakeddit_overfitting.csv")
print("\n"+"="*70+"\n  ✅ Complete!\n"+"="*70)